# UTS Praktikum AST — Paket 5: Clustering DBSCAN dan OPTICS

**Dataset:** `Kasus_5.csv` (kolom `koordinat_x`, `koordinat_y`, n = 147 titik layanan)

**Konteks:** Dinas pelayanan ingin mengelompokkan titik layanan berdasarkan kedekatan koordinat dan mendeteksi lokasi terpencil (noise/outlier).

**Isi notebook (sesuai instruksi Paket 5):**
- **Tahap I (tertulis):** hitung jarak Euclidean dua pasangan titik pertama + jelaskan `eps`, `min_samples`, *core point*, *noise*.
- **Tahap II (komputer):** scatter plot, standardisasi, DBSCAN + OPTICS, jumlah cluster & noise, silhouette (hanya bila valid), perbandingan + parameter paling berpengaruh, kesimpulan.

> Cara pakai di Colab: upload `Kasus_5.csv` bersama notebook ini, lalu Run All.

## TAHAP I — Jawaban tertulis (disalin ke kertas)

### 1. Perhitungan jarak Euclidean dua pasangan titik pertama

Rumus: d(p,q) = sqrt((xp-xq)^2 + (yp-yq)^2)

Data:
- Titik 1: (4.122, 5.975)
- Titik 2: (5.120, 2.053)
- Titik 3: (1.750, 1.776)
- Titik 4: (1.288, 4.802)

**Pasangan 1 — Titik 1 vs Titik 2:**
 dx = 4.122-5.120 = -0.998 -> dx^2 = 0.996004
 dy = 5.975-2.053 = 3.922 -> dy^2 = 15.382084
 jumlah = 16.378088
 d(1,2) = sqrt(16.378088) = 4.047 (3 desimal)

**Pasangan 2 — Titik 3 vs Titik 4:**
 dx = 1.750-1.288 = 0.462 -> dx^2 = 0.213444
 dy = 1.776-4.802 = -3.026 -> dy^2 = 9.156676
 jumlah = 9.370120
 d(3,4) = sqrt(9.370120) = 3.061 (3 desimal)

**Jawaban: d12 = 4.047, d34 = 3.061**

### 2. Konsep DBSCAN

- `eps`: radius lingkungan. Dua titik bertetangga jika jarak <= eps. eps kecil -> cluster pecah & banyak noise; eps besar -> cluster menggabung, noise hilang. Parameter paling berpengaruh.
- `min_samples`: jumlah minimum titik dalam radius eps agar disebut core point. Besar -> ketat (noise bertambah); kecil -> longgar.
- *Core point*: punya >= min_samples titik dalam eps (termasuk dirinya). Biji cluster.
- *Border point*: bukan core tapi dalam eps dari core. Anggota tapi tidak memperluas cluster.
- *Noise* (label -1): bukan core dan tidak terjangkau core mana pun. Inilah lokasi terpencil yang dicari dinas.

In [ ]:
# Verifikasi Tahap I
import math
p1=(4.122,5.975); p2=(5.120,2.053); p3=(1.750,1.776); p4=(1.288,4.802)
def euclid(p,q):
    dx=p[0]-q[0]; dy=p[1]-q[1]
    return dx,dy,math.sqrt(dx**2+dy**2)
for nama,a,b in [('Titik1-Titik2',p1,p2),('Titik3-Titik4',p3,p4)]:
    dx,dy,d=euclid(a,b)
    print(f'{nama}: dx={dx:.3f} (dx^2={dx**2:.6f}), dy={dy:.3f} (dy^2={dy**2:.6f}), d={d:.6f} -> {d:.3f}')

## TAHAP II — Kerja komputer
Instruksi paket: buat scatter plot, lakukan standardisasi bila diperlukan, terapkan DBSCAN dan OPTICS, tampilkan jumlah cluster serta noise, evaluasi silhouette hanya bila valid, bandingkan hasil dan jelaskan parameter yang paling berpengaruh.

In [ ]:
# 0. Setup & load data
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
kandidat=['Kasus_5.csv','/content/Kasus_5.csv','/home/herdan/UTSAST/Kasus_5.csv','/tmp/opencode/UTSAST/Kasus_5.csv']
path=next((p for p in kandidat if os.path.exists(p)),None)
print('CSV path:',path)
if path is None:
    try:
        from google.colab import files
        print('Upload Kasus_5.csv ...')
        files.upload()
        path='Kasus_5.csv'
    except ImportError:
        raise FileNotFoundError('Letakkan Kasus_5.csv sejajar notebook.')
df=pd.read_csv(path)
print('Bentuk data:',df.shape)
print(df.head(6).to_string(index=False))
print()
print(df.describe().round(3).to_string())
print('\nMissing:')
print(df.isna().sum().to_string())

In [ ]:
# 1. Scatter plot + histogram (EDA)
fig,ax=plt.subplots(1,2,figsize=(12,4.5))
ax[0].scatter(df['koordinat_x'],df['koordinat_y'],s=28,alpha=0.8)
ax[0].set_title('Scatter titik layanan (mentah)')
ax[0].set_xlabel('koordinat_x');ax[0].set_ylabel('koordinat_y');ax[0].grid(alpha=0.3)
ax[1].hist(df['koordinat_x'],bins=20,alpha=0.6,label='x')
ax[1].hist(df['koordinat_y'],bins=20,alpha=0.6,label='y')
ax[1].set_title('Distribusi x vs y');ax[1].set_xlabel('nilai');ax[1].set_ylabel('frekuensi')
ax[1].legend();ax[1].grid(alpha=0.3)
plt.tight_layout();plt.show()
print('Interpretasi: 3 gugus padat (kiri-bawah x~2,y~2; kanan-bawah x~6,y~2; atas x~4-5,y~6) + titik terpencil (mis. 7.6,4.5). Cocok untuk density-based.')

In [ ]:
# 2. Standardisasi
from sklearn.preprocessing import StandardScaler
print('STD mentah:',df[['koordinat_x','koordinat_y']].std().round(3).to_dict())
print('Mean mentah:',df[['koordinat_x','koordinat_y']].mean().round(3).to_dict())
scaler=StandardScaler()
X_scaled=scaler.fit_transform(df[['koordinat_x','koordinat_y']])
print('STD sesudah:',X_scaled.std(axis=0).round(3))
fig,ax=plt.subplots(1,2,figsize=(12,4.5))
ax[0].scatter(df['koordinat_x'],df['koordinat_y'],s=28,alpha=0.7)
ax[0].set_title('Mentah');ax[0].set_xlabel('x');ax[0].set_ylabel('y');ax[0].grid(alpha=0.3)
ax[1].scatter(X_scaled[:,0],X_scaled[:,1],s=28,alpha=0.7,color='darkorange')
ax[1].set_title('Terstandardisasi (mean=0,std=1)');ax[1].set_xlabel('x_scaled');ax[1].set_ylabel('y_scaled');ax[1].grid(alpha=0.3)
plt.tight_layout();plt.show()
print('Keputusan: std x,y mirip (1.78 vs 1.97) jd standardisasi tidak wajib, TETAP dipakai agar eps konsisten.')

In [ ]:
# 3. Heuristik eps: k-distance plot (k=min_samples)
from sklearn.neighbors import NearestNeighbors
k=5
nn=NearestNeighbors(n_neighbors=k+1).fit(X_scaled)
dist,_=nn.kneighbors(X_scaled)
k_dist=np.sort(dist[:,k])
plt.figure(figsize=(7,4.5))
plt.plot(k_dist)
plt.title(f'{k}-distance plot - cari siku untuk eps')
plt.xlabel('titik terurut');plt.ylabel(f'jarak tetangga ke-{k}');plt.grid(alpha=0.3);plt.show()
print(f'p90={np.percentile(k_dist,90):.3f}, p95={np.percentile(k_dist,95):.3f}, maks={k_dist.max():.3f}')
print('Siku di 0.3-0.5 -> eps=0.4 baseline (uji 0.3/0.4/0.5).')

In [ ]:
# 4. DBSCAN terstandardisasi
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
def ringkas(label,nama):
    label=np.asarray(label)
    nc=len(set(label))-(1 if -1 in label else 0)
    nz=int((label==-1).sum())
    sil=float('nan'); valid=(nc>=2)and(nz<len(label)-1)
    if valid:
        try:
            sil=silhouette_score(X_scaled[label!=-1],label[label!=-1])
        except Exception as e:
            print('silhouette gagal:',e); valid=False
    print(f'{nama}: cluster={nc}, noise={nz}, silhouette={sil:.3f} (valid={valid})')
    vals,cnts=np.unique(label,return_counts=True)
    print(' distribusi:',dict(zip(vals.tolist(),cnts.tolist())))
    return nc,nz,sil
hasil_db={}
for eps in [0.3,0.4,0.5]:
    lab=DBSCAN(eps=eps,min_samples=5).fit_predict(X_scaled)
    hasil_db[eps]=lab
    ringkas(lab,f'DBSCAN eps={eps}, min_samples=5')

In [ ]:
# 5. Visualisasi DBSCAN final + daftar noise (koordinat asli)
lab_db=hasil_db[0.4]
ringkas(lab_db,'DBSCAN final eps=0.4 ms=5')
plt.figure(figsize=(7,5.5))
unik=sorted(set(lab_db))
colors=plt.cm.tab10(np.linspace(0,1,max(len(unik),10)))
for k2,c in zip(unik,colors):
    m=lab_db==k2
    nama='noise' if k2==-1 else f'cluster {k2}'
    plt.scatter(X_scaled[m,0],X_scaled[m,1],s=60 if k2==-1 else 35,label=f'{nama} (n={m.sum()})',marker='x' if k2==-1 else 'o',alpha=0.85)
plt.title('DBSCAN eps=0.4, min_samples=5 (terstandardisasi)')
plt.xlabel('x_scaled');plt.ylabel('y_scaled');plt.legend();plt.grid(alpha=0.3);plt.show()
noise_df=df.loc[lab_db==-1].reset_index(drop=True)
print(f'Jumlah noise DBSCAN: {len(noise_df)}')
print(noise_df.round(3).to_string())

In [ ]:
# 6. OPTICS + reachability plot
from sklearn.cluster import OPTICS
opt_a=OPTICS(min_samples=5,xi=0.05,min_cluster_size=0.05).fit(X_scaled)
lab_opt_a=opt_a.labels_
ringkas(lab_opt_a,'OPTICS ms=5 xi=0.05')
opt_b=OPTICS(min_samples=5,max_eps=0.4,cluster_method='dbscan').fit(X_scaled)
lab_opt_b=opt_b.labels_
ringkas(lab_opt_b,'OPTICS-dbscan max_eps=0.4')
space=np.arange(len(X_scaled))
ordering=opt_a.ordering_
reach=opt_a.reachability_[ordering]
finite=reach[np.isfinite(reach)]
reach_plot=np.where(np.isinf(reach),finite.max() if len(finite)>0 else 0,reach)
plt.figure(figsize=(9,4))
plt.plot(space,reach_plot,lw=1)
plt.title('OPTICS reachability plot (ms=5) - lembah=cluster')
plt.xlabel('urutan OPTICS');plt.ylabel('reachability');plt.grid(alpha=0.3);plt.show()
print('Lembah dalam & lebar = cluster padat; puncak = batas/noise.')

In [ ]:
# 7. Perbandingan berdampingan + tabel rekap
fig,ax=plt.subplots(1,3,figsize=(16,4.8),sharex=True,sharey=True)
for a,lab,judul in [(ax[0],lab_db,'DBSCAN (eps=0.4,ms=5)'),(ax[1],lab_opt_a,'OPTICS (ms=5,xi=0.05)'),(ax[2],lab_opt_b,'OPTICS-dbscan (eps=0.4)')]:
    for k2 in sorted(set(lab)):
        m=lab==k2
        a.scatter(X_scaled[m,0],X_scaled[m,1],s=30,label=f'{"noise" if k2==-1 else f"C{k2}"}:{m.sum()}',marker='x' if k2==-1 else 'o',alpha=0.85)
    nc=len(set(lab))-(1 if -1 in lab else 0)
    a.set_title(f'{judul}\n({nc} cluster,{(lab==-1).sum()} noise)')
    a.set_xlabel('x_scaled');a.legend(fontsize=7);a.grid(alpha=0.3)
ax[0].set_ylabel('y_scaled')
plt.suptitle('Perbandingan DBSCAN vs OPTICS',y=1.02)
plt.tight_layout();plt.show()
rekap=[]
for nama,lab in [('DBSCAN eps=0.4',lab_db),('OPTICS xi=0.05',lab_opt_a),('OPTICS-dbscan 0.4',lab_opt_b)]:
    nc=len(set(lab))-(1 if -1 in lab else 0)
    nz=int((lab==-1).sum())
    try:
        s=silhouette_score(X_scaled[lab!=-1],lab[lab!=-1]) if nc>=2 else float('nan')
    except Exception:
        s=float('nan')
    rekap.append((nama,nc,nz,round(float(s),3)))
print(pd.DataFrame(rekap,columns=['metode','n_cluster','n_noise','silhouette_tanpa_noise']).to_string(index=False))

## Kesimpulan & interpretasi

1. **Hasil baseline (terstandardisasi):** DBSCAN (eps=0.4, ms=5) = **3 cluster + 6 noise, silhouette ~0.79** (valid, >=2 cluster). Tiga zona: kiri-bawah, kanan-bawah, atas.
2. **OPTICS (ms=5, xi=0.05)** memecah menjadi cluster kecil lebih banyak dengan **noise jauh lebih besar**; silhouette tanpa-noise bisa tampak tinggi tapi tidak sebanding langsung karena dihitung pada subset berbeda. OPTICS mode dbscan (max_eps=0.4) **identik DBSCAN** — bukti OPTICS generalisasi DBSCAN.
3. **Silhouette hanya bila valid:** dihitung tanpa noise dan hanya jika >=2 cluster; jika tidak valid jangan dilaporkan.
4. **Parameter paling berpengaruh:** DBSCAN = **eps** (0.3->0.5: noise 10->5) lalu min_samples (5->10: noise 6->7); OPTICS = **xi & min_cluster_size**; pra-proses = **standardisasi** menentukan makna eps.
5. **Rekomendasi dinas:** pakai **DBSCAN eps=0.4 ms=5** untuk peta 3 zona + daftar lokasi terpencil (tabel noise) untuk inspeksi; OPTICS + reachability untuk analisis hierarki/sub-zona.
6. **Keterbatasan:** sensitif eps/xi; silhouette abaikan noise; titik terluar (7.645,4.520 & 7.406,5.580) selalu noise — validasi lapangan.

In [ ]:
# Lampiran sensitivitas
print('--- DBSCAN ---')
for eps in [0.3,0.4,0.5,0.6]:
    for ms in [5,10]:
        lab=DBSCAN(eps=eps,min_samples=ms).fit_predict(X_scaled)
        nc=len(set(lab))-(1 if -1 in lab else 0)
        print(f'eps={eps} ms={ms}: cluster={nc}, noise={(lab==-1).sum()}')
print()
print('--- OPTICS (xi=0.05) ---')
for ms in [5,8,10]:
    lab=OPTICS(min_samples=ms,xi=0.05).fit_predict(X_scaled)
    nc=len(set(lab))-(1 if -1 in lab else 0)
    print(f'ms={ms}: cluster={nc}, noise={(lab==-1).sum()}')